# Part 6 Demo — Supervised AI Classifier

This notebook demonstrates the Part 6 classifier on a synthetic feature catalog. Replace the synthetic catalog with the organizer-provided curated labeled catalog for real training.

In [ ]:
from pathlib import Path
import pandas as pd

from exoplanet_pipeline.ml_synthetic import make_synthetic_ml_feature_catalog
from exoplanet_pipeline.ml import train_ai_classifier, predict_ai_classifier, write_training_artifacts
from exoplanet_pipeline.ml_diagnostics import plot_confusion_matrix, plot_feature_importance, plot_prediction_probability_bars

In [ ]:
out_dir = Path("outputs_part6_ai")
out_dir.mkdir(exist_ok=True)

catalog = make_synthetic_ml_feature_catalog(n_per_class=45, random_seed=2026)
catalog.head()

In [ ]:
catalog["label"].value_counts()

In [ ]:
result = train_ai_classifier(
    catalog,
    label_col="label",
    model_type="random_forest",
    calibrate=False,
    test_size=0.25,
    random_state=2026,
)

{k: v for k, v in result.test_metrics.items() if k != "classification_report"}

In [ ]:
result.feature_importance.head(20)

In [ ]:
paths = write_training_artifacts(result, out_dir)
plot_confusion_matrix(result.confusion_matrix, result.class_names, out_dir / "notebook_confusion_matrix.png")
plot_feature_importance(result.feature_importance, out_dir / "notebook_feature_importance.png")
paths

In [ ]:
predict_input = catalog.sample(n=12, random_state=7).drop(columns=["label"])
predictions = predict_ai_classifier(result.model_bundle, predict_input)
predictions[["tic_id", "ai_predicted_class", "ai_confidence", "final_predicted_class", "final_confidence", "final_classifier_warnings"]].head(12)

In [ ]:
plot_prediction_probability_bars(predictions, out_dir / "notebook_prediction_probabilities.png")
print("Wrote Part 6 outputs to", out_dir.resolve())

## How to use with the real curated dataset

Use:

```bash
python scripts/train_ai_classifier_from_catalog.py curated_labeled_catalog.csv --label-col label --output-dir outputs_part6_ai
```

Then predict science candidates with:

```bash
python scripts/predict_ai_classifier_catalog.py outputs_part6_ai/part6_ai_classifier.joblib candidate_catalog.csv --output-csv outputs_part6_ai/science_predictions.csv
```